# episodata: a hands-on tour

A runnable companion to [`docs/getting_started.md`](../docs/getting_started.md).
Build a dataset, read it, sample it two ways, persist it, and grow it online
two ways.

In [1]:
import numpy as np

from episodata import Dataset

## 1. Build a toy dataset

An episode is a plain dict: `initial_observation` (what `env.reset()`
returned), then `observations` / `actions` / `rewards` with one entry per
step, plus the Gymnasium-style `terminated` / `truncated` signals. All
lengths count env steps — there is no dummy value to construct anywhere.
`Dataset.from_episodes` infers a schema from the first episode — no schema
to write by hand.

In [2]:
def make_episode(length: int, seed: int, terminated: bool = True) -> dict:
    """``length`` env steps: ``initial_observation`` plus equal-length
    observations/actions/rewards — one entry per step."""
    rng = np.random.default_rng(seed)
    return {
        "initial_observation": {
            "front_camera": rng.integers(0, 256, size=(3, 32, 32), dtype=np.uint8),
            "state": rng.standard_normal(6).astype(np.float32),
        },
        "observations": {
            "front_camera": rng.integers(0, 256, size=(length, 3, 32, 32), dtype=np.uint8),
            "state": rng.standard_normal((length, 6)).astype(np.float32),
        },
        "actions": rng.standard_normal((length, 3)).astype(np.float32),
        "rewards": rng.standard_normal(length).astype(np.float32),
        "terminated": terminated,
        "truncated": False,
    }

episodes = [
    make_episode(50, seed=0),
    make_episode(30, seed=1),
    make_episode(12, seed=2, terminated=False),  # still ongoing
]

dataset = Dataset.from_episodes(episodes)
dataset

Dataset(backend='memory', num_episodes=3, fields=['front_camera', 'state', 'action', 'reward'])

In [3]:
dataset.schema

DatasetSchema(spaces=['image', 'vector', 'reward', 'action'], fields=['front_camera', 'state', 'action', 'reward'])

## 2. Reading: episodes and segments

`dataset.episode(i)` is a lazy view — nothing is read until you ask for a
segment. `episode.segment(start, stop)` returns transitions `[start, stop)`
as a `Segment`. Access is role-first: `seg.observation` (aliases `seg.obs`,
`seg.observations`) is the obs each action was taken at, `seg.next_observation`
(`seg.next_obs`) the obs it produced, and `terminated` / `truncated` /
`mask` are per-transition flags. A role holding one bare array — like the
action here — resolves straight to that array. The arrays are zero-copy
views into one shared row buffer, and a segment starting at 0 surfaces the
reset observation as `seg.obs[k][0]`.

In [4]:
episode = dataset.episode(0)
seg = episode.segment(0, 5, fields=["front_camera", "state", "action"])

seg.obs["front_camera"].shape, seg.next_obs["front_camera"].shape, seg.action.shape, seg.terminated

((5, 3, 32, 32),
 (5, 3, 32, 32),
 (5, 3),
 array([False, False, False, False, False]))

## 3. Fields: flat, space, and role access

Fields group into **spaces** (shared shape/dtype — `image`, `vector`, ...)
and into **roles** (`observations`, `actions`, `rewards`, `infos`). Flat keys,
space attributes, and role views all read the same underlying data — use
whichever is clearest at the call site.

In [5]:
print("field:     ", seg.obs.front_camera.shape)
print("space:     ", seg.obs.space("image").front_camera.shape)
print("role view: ", list(seg.obs))

for key, value in seg.obs.space("image").items():
    print(f"image {key}: {value.shape}")

field:      (5, 3, 32, 32)
space:      (5, 3, 32, 32)
role view:  ['front_camera', 'state']
image front_camera: (5, 3, 32, 32)


## 4. Sampling for training

### Map-style: `segments()` + `DataLoader`

`dataset.segments(...)` is an indexable, map-style view over fixed-length
segments — plain `len()` / `[i]`, so it plugs directly into
`torch.utils.data.DataLoader` for `num_workers` read parallelism (each
worker decompresses its own share of episodes independently). `episodata`
itself never imports torch: `segments[i]` returns a `Segment` and works with
no torch installed at all.

In [ ]:
from torch.utils.data import DataLoader

segments = dataset.segments(
    fields=["front_camera", "state", "action", "reward"], sequence_length=8
)
loader = DataLoader(
    segments, batch_size=8, shuffle=True, num_workers=0, collate_fn=segments.collate,
)
batch = next(iter(loader))
batch.obs.front_camera.shape   # episodata.Batch, arrays [B, L, ...]

### Custom samplers: prioritized replay

`segments()` is a plain map-style dataset, so *any* `torch.utils.data.Sampler`
works over it — `DataLoader`'s `sampler` argument replaces `shuffle` with your
own draw order. Prioritization itself is entirely outside episodata: compute
priorities however you like (TD-error, recency, ...) and hand `DataLoader` a
`Sampler` that draws indices accordingly. Here we weight by a stand-in
"priority" — each segment's absolute reward sum:

In [ ]:
from torch.utils.data import Sampler


class PrioritizedSampler(Sampler):
    """Draws segment indices with replacement, weighted by external priorities."""

    def __init__(self, priorities: np.ndarray, num_samples: int, seed: int | None = None):
        self.priorities = np.asarray(priorities, dtype=np.float64)
        self.num_samples = num_samples
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        probs = self.priorities / self.priorities.sum()
        return iter(self.rng.choice(len(self.priorities), size=self.num_samples, p=probs).tolist())

    def __len__(self) -> int:
        return self.num_samples


priorities = np.array([np.abs(segments[i].reward).sum() for i in range(len(segments))])
sampler = PrioritizedSampler(priorities, num_samples=len(segments), seed=0)
prioritized_loader = DataLoader(
    segments, batch_size=8, sampler=sampler, collate_fn=segments.collate,
)
prioritized_batch = next(iter(prioritized_loader))
prioritized_batch.obs.front_camera.shape

Recompute `priorities` and rebuild the sampler as often as your algorithm
needs (each step, each epoch, ...) — episodata only supplies the indexable
segments; how they're drawn is entirely up to the caller.

### Streaming: `segment_stream()`

For a simpler infinite, shuffled, single-process stream — no `DataLoader`
needed — with optional `context` / `target` splitting for world-model
training:

In [8]:
stream = dataset.segment_stream(
    fields=["front_camera", "state", "action"],
    context_length=4,
    target_length=8,
    batch_size=16,
    seed=0,
)
stream_batch = stream.sample()

print("batch:  ", stream_batch.obs.front_camera.shape)     # [B, L, ...]
print("context:", stream_batch.context.obs.state.shape)    # [B, context_length, ...]
print("target: ", stream_batch.target.obs.state.shape)     # [B, target_length, ...]

batch:   (16, 12, 3, 32, 32)
context: (16, 4, 6)
target:  (16, 8, 6)


For control, `sample_transitions` draws `(s, a, r, s', done)` pairs directly
— no manual time-shifting.

In [9]:
transitions = dataset.sample_transitions(batch_size=8, seed=0)

transitions.obs["front_camera"].shape, transitions.reward.shape, transitions.terminated

((8, 3, 32, 32),
 (8,),
 array([False, False, False, False, False, False, False, False]))

## 5. Persistence: same API, on disk

Add a `path` to persist to the `npz_directory` backend (the default once a
path is given); `Dataset.open` reopens it — same schema, same calls, no
re-inference.

In [10]:
import shutil
import tempfile

path = tempfile.mkdtemp(prefix="episodata_")
shutil.rmtree(path)  # from_episodes creates the directory itself

Dataset.from_episodes(episodes, path=path)
reopened = Dataset.open(path)
reopened

Dataset(backend='npz_directory', num_episodes=3, fields=['front_camera', 'state', 'action', 'reward'])

Outgrown one-file-per-episode? `dataset.copy_to(new_path, backend="zarr")`
streams the dataset across the storage boundary onto the chunked `zarr`
backend — same read/write API on the other side.

## 6. Collecting data online, two ways

The write API mirrors a Gymnasium rollout one-to-one: `new_episode` records
what `env.reset()` returned (the initial observation), then one `add_step`
call per `env.step`. The usual way is a writer, kept for the lifetime of
the rollout:

In [11]:
writer = dataset.new_episode({
    "front_camera": np.zeros((3, 32, 32), dtype=np.uint8),
    "state": np.zeros(6, dtype=np.float32),
})

for t in range(5):
    writer.add_step({
        "observations": {
            "front_camera": np.zeros((3, 32, 32), dtype=np.uint8),
            "state": np.zeros(6, dtype=np.float32),
        },
        "actions": np.zeros(3, dtype=np.float32),
        "rewards": 1.0,
        "terminated": t == 4,   # a True signal finalizes the episode
        "truncated": False,
    })

dataset.episode(writer.episode_id).terminated

True

A writer is a stateless handle — only its `episode_id` needs to be kept. The
same operations exist directly on `Dataset` by id, useful when you'd rather
not carry a writer object around (e.g. across process boundaries), or want
to append a whole segment in one call instead of step by step:

In [12]:
episode_id = dataset.new_episode({  # writer discarded; only the id is kept
    "front_camera": np.zeros((3, 32, 32), dtype=np.uint8),
    "state": np.zeros(6, dtype=np.float32),
}).episode_id

L = 5
dataset.add_steps(episode_id, {
    "observations": {
        "front_camera": np.zeros((L, 3, 32, 32), dtype=np.uint8),
        "state": np.zeros((L, 6), dtype=np.float32),
    },
    "actions": np.zeros((L, 3), dtype=np.float32),
    "rewards": np.ones(L, dtype=np.float32),
    "terminated": np.array([False, False, False, False, True]),
    "truncated": np.zeros(L, dtype=bool),
})

dataset.episode(episode_id).terminated, dataset.num_episodes

(True, 5)

## Next steps

- [`docs/getting_started.md`](../docs/getting_started.md) — the design in
  three layers, and how everything counts env steps.
- [README](../README.md) — full reference: hierarchical fields, schema
  refinement, backend internals, action-out conversion.